# Contrastive training Mini-CLIP modela

U ovom delu implementiran je trening Mini-CLIP modela korišćenjem
**contrastive learning** pristupa.

Model se sastoji iz dva enkodera:
- image encoder-a, zasnovanog na ResNet18 arhitekturi;
- text encoder-a, zasnovanog na Transformer arhitekturi.

Oba enkodera mapiraju svoje ulaze u zajednički embedding prostor dimenzije 256.
Cilj treninga je da embedding odgovarajuće slike i njenog tekstualnog opisa
bude što sličniji, dok embedding-i nepovezanih image-caption parova treba
da budu međusobno manje slični.

Prvo se koristi ResNet18 implementiran od početka kao
image encoder i Transformer kao text encoder. Kasnije se isti postupak
ponavlja sa pretrained i fine-tuned ResNet18 varijantama.

In [1]:
import os
import json

import torch
import torch.nn.functional as F

from image_encoder import (
    ResNet18Encoder,
    ResNet18EncoderPretrained,
    ResNet18EncoderFineTuned
)

from text_encoder import MiniTextTransformer

In [2]:
%%capture
get_ipython().run_line_magic("run", "01_data_setup.ipynb")

### Učitavanje pripremljenih podataka

Dataset i DataLoader-i za contrastive training formirani su u
`01_data_setup.ipynb`. U ovoj svesci koriste se već pripremljeni
image-caption batch-evi, kako bi fokus ostao na generisanju embedding-a,
contrastive loss-u i treningu modela.

In [3]:
train_loader = contrastive_train_loader
val_loader = contrastive_val_loader

### Generisanje image i text embedding-a

Jedan batch se prosleđuje kroz oba encoder-a kako bi se dobile njihove
reprezentacije u zajedničkom embedding prostoru.

Slike iz batch-a prosleđuju se image encoder-u, dok se `input_ids` i
`attention_mask` vrednosti prosleđuju text encoder-u.

Pošto oba encoder-a vraćaju embedding vektore dimenzije 256, dobijene
reprezentacije mogu kasnije direktno da se porede pomoću kosinusne sličnosti.

In [4]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = "./data"
VOCAB_FILE = os.path.join(DATA_DIR, "processed", "vocab.json")

with open(VOCAB_FILE, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

token_to_id = vocab_data["token_to_id"]

VOCAB_SIZE = len(token_to_id)
MAX_LENGTH = vocab_data["max_length"]
PAD_ID = token_to_id["<PAD>"]

print("Vocabulary size:", VOCAB_SIZE)
print("Maximum caption length:", MAX_LENGTH)
print("PAD ID:", PAD_ID)

Vocabulary size: 7747
Maximum caption length: 32
PAD ID: 0


In [5]:
EMBEDDING_DIM = 256

image_encoder = ResNet18Encoder(
    emb_dim=EMBEDDING_DIM
).to(DEVICE)

text_encoder = MiniTextTransformer(
    vocab_size=VOCAB_SIZE,
    max_length=MAX_LENGTH,
    padding_idx=PAD_ID,
    output_dim=EMBEDDING_DIM
).to(DEVICE)

In [6]:
batch = next(iter(train_loader))

images = batch["image"].to(DEVICE)
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)

image_embeddings = image_encoder(images)

text_embeddings = text_encoder(
    input_ids,
    attention_mask
)

print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape)

Image embeddings: torch.Size([8, 256])
Text embeddings: torch.Size([8, 256])


### Računanje sličnosti između image i text embedding-a

Nakon što su slike i caption-i mapirani u zajednički embedding prostor,
računa se sličnost između svakog image embedding-a i svakog text embedding-a
u batch-u.

Pošto su embedding-i normalizovani, njihov skalarni proizvod odgovara
kosinusnoj sličnosti.

Rezultat je matrica dimenzije `batch_size × batch_size`, gde element
na poziciji `(i, j)` predstavlja sličnost između i-te slike i j-tog caption-a.

In [7]:
similarities = image_embeddings @ text_embeddings.T

print("Similarity matrix shape:", similarities.shape)
print(similarities) # cilj je tokom treninga da dijagonala bude što veća

Similarity matrix shape: torch.Size([8, 8])
tensor([[-0.0448, -0.0449, -0.0367, -0.0522, -0.0783, -0.0840, -0.0455, -0.0531],
        [-0.0838, -0.0709, -0.0617, -0.0796, -0.0922, -0.1077, -0.0745, -0.0954],
        [-0.0640, -0.0615, -0.0486, -0.0715, -0.0943, -0.0942, -0.0674, -0.0789],
        [-0.0879, -0.0671, -0.0496, -0.0845, -0.0830, -0.1048, -0.0740, -0.0976],
        [-0.0666, -0.0638, -0.0444, -0.0649, -0.0876, -0.1014, -0.0612, -0.0774],
        [-0.0645, -0.0601, -0.0435, -0.0612, -0.0857, -0.1029, -0.0625, -0.0750],
        [-0.0881, -0.0755, -0.0685, -0.0809, -0.0940, -0.1015, -0.0751, -0.0971],
        [-0.0900, -0.0736, -0.0737, -0.0872, -0.0933, -0.1069, -0.0808, -0.0982]],
       grad_fn=<MmBackward0>)


### Definisanje pozitivnih parova

U svakom batch-u i-ta slika i i-ti caption predstavljaju odgovarajući
image-caption par.

Zbog toga su pozitivni parovi smešteni na dijagonali matrice sličnosti.
Za svaku sliku formira se labela koja označava indeks njenog odgovarajućeg
caption-a.

Na primer, za batch veličine 8 labele su `[0, 1, 2, ..., 7]`, što znači
da je za sliku na poziciji 0 odgovarajući caption na poziciji 0,
za sliku na poziciji 1 caption na poziciji 1, itd.

In [8]:
labels = torch.arange(
    similarities.size(0),
    device=DEVICE
)

print(labels)

tensor([0, 1, 2, 3, 4, 5, 6, 7])


### Image-to-text contrastive loss

Za svaku sliku u batch-u model treba da prepozna koji caption joj pripada.

Svaki red matrice sličnosti posmatra se kao skup mogućih caption-a za jednu
sliku. Tačan caption nalazi se na istoj poziciji kao i slika, što je određeno
prethodno formiranim `labels`.

Cross-entropy loss kažnjava model ukoliko odgovarajući caption nema veću
sličnost sa slikom od ostalih caption-a u batch-u.

In [9]:
image_to_text_loss = F.cross_entropy(
    similarities,
    labels
)

print("Image-to-text loss:", image_to_text_loss.item())

Image-to-text loss: 2.08048152923584


### Simetrični contrastive loss

Pored image-to-text poređenja, loss se računa i u suprotnom smeru.

Kod text-to-image dela svaki caption treba da prepozna odgovarajuću sliku
među svim slikama u batch-u. Za to se koristi transponovana matrica
sličnosti.

Konačni contrastive loss dobija se kao prosek image-to-text i
text-to-image loss-a, čime se oba encoder-a uče da međusobno usklade
svoje embedding reprezentacije.

In [10]:
text_to_image_loss = F.cross_entropy(
    similarities.T,
    labels
)

contrastive_loss = (
    image_to_text_loss + text_to_image_loss
) / 2

print("Image-to-text loss:", image_to_text_loss.item())
print("Text-to-image loss:", text_to_image_loss.item())
print("Contrastive loss:", contrastive_loss.item())

Image-to-text loss: 2.08048152923584
Text-to-image loss: 2.080436944961548
Contrastive loss: 2.0804591178894043
